# Scoring
> Score a sampling campaign against the ground truth

Work in progress: experiments for the per-unit table in `scoring-plan.md` (#12).

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
import geopandas as gpd
from fastcore.test import *

from trufl.utils import gridder, zonal_values
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, MoranICB, PriorCB
from trufl.optimizer import Optimizer

One campaign to experiment on: a 10 x 10 grid over the simulated raster, with between 0 and 9 samples per unit, so some units have none:

In [ ]:
fname_raster = 'files/ground-truth-01-4326-simulated.tif'
gdf_grid = gridder(fname_raster, nrows=10, ncols=10)
n = np.random.default_rng(0).integers(0, 10, size=len(gdf_grid))
samples = DataCollector(fname_raster).collect(Sampler(gdf_grid).sample(n, method='uniform', rng=0))
samples.head()

`unit_stats` compares each unit's samples with the raster cells inside it. Units without samples have a count of 0 and `NaN` for the measured values and the distance:

In [ ]:
def unit_stats(
    samples:gpd.GeoDataFrame, # Measurements indexed by `loc_id`, with a `value` column
    gdf_units:gpd.GeoDataFrame, # Units indexed by `loc_id`
    fname_raster:str, # Ground-truth raster
) -> pd.DataFrame: # One row per unit
    "True and measured mean and maximum, sample count and Wasserstein distance of each unit."
    rows = {}
    for loc_id, geom in gdf_units.geometry.items():
        true = zonal_values(fname_raster, geom)
        meas = samples.value[samples.index == loc_id].values
        rows[loc_id] = dict(true_mean=true.mean(), true_max=true.max(),
                            meas_mean=meas.mean() if len(meas) else np.nan, meas_max=meas.max() if len(meas) else np.nan,
                            n=len(meas), wasserstein=wasserstein_distance(true, meas) if len(meas) else np.nan)
    return pd.DataFrame.from_dict(rows, orient='index').rename_axis('loc_id')

In [ ]:
df = unit_stats(samples, gdf_grid, fname_raster)
test_eq(df.n.sum(), len(samples))
df.head()

## Multi-year campaign

The campaign of [Run a campaign over several years](how-to/01_multi_year_campaign.ipynb), with fixed seeds: 150 samples a year for four years, ranked from the prior in the first year and from six state variables after that. `tables` keeps the per-unit table after each year:

In [ ]:
benefit_criteria = [True, True, True, False, False, True]
weights = [0.2, 0.1, 0.1, 0.2, 0.2, 0.2]
np.random.seed(0)
dc = DataCollector(fname_raster)
sampler = Sampler(gdf_grid)
state = State(samples.iloc[:0], gdf_grid, cbs=[PriorCB(fname_raster)])()
df_rank = Optimizer(state=state).get_rank(is_benefit_x=[True], w_vector=[1], n_method=None, c_method=None, w_method=None, s_method="CP")
yearly, tables = [], {}
for year in range(1, 5):
    n = rank_to_sample(df_rank['rank'].sort_index().values, budget=150, min=1, policy="quantiles")
    yearly.append(dc.collect(sampler.sample(n, method='uniform', rng=year)))
    sofar = pd.concat(yearly)
    tables[year] = unit_stats(sofar, gdf_grid, fname_raster)
    state = State(sofar, gdf_grid, cbs=[MaxCB(), MinCB(), StdCB(), CountCB(), MoranICB(k=5), PriorCB(fname_raster)])()
    df_rank = Optimizer(state=state).get_rank(is_benefit_x=benefit_criteria, w_vector=weights, n_method="LINEAR1", c_method=None, w_method=None, s_method="CP")

How the table changes from year to year, summarised over units:

In [ ]:
pd.DataFrame({year: {'samples': t.n.sum(), 'unsampled': (t.n == 0).sum(),
                     'mean_abs_err_mean': (t.meas_mean - t.true_mean).abs().mean(),
                     'mean_err_max': (t.meas_max - t.true_max).mean(),
                     'mean_wasserstein': t.wasserstein.mean()}
              for year, t in tables.items()}).round(4)